In [7]:
import pandas as pd
!pip install streamlit -q

#df = pd.read_csv('Resume.csv', engine='python')

df = pd.read_csv('Resume.csv')

print(df.head())
print(df['Category'].value_counts())

         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118


In [8]:
#Extracting Skills & Keywords
import re
# Example master skill dictionary (you can expand this)
SKILL_DB = [
    'python', 'java', 'c++', 'machine learning', 'deep learning',
    'sql', 'pandas', 'numpy', 'scikit-learn', 'react', 'node.js',
    'mongodb', 'html', 'css', 'javascript', 'flask', 'streamlit'
]

def extract_skills(text, skill_database):
  text_lower = text.lower()
  found_skills = [skill for skill in skill_database if re.search(r'\b' + re.escape(skill) + r'\b', text_lower)]
  return list(set(found_skills))

df['extracted_skills'] = df['Resume_str'].apply(lambda x: extract_skills(x, SKILL_DB))

print(df[['Resume_str', 'extracted_skills']].head())
#print(df.head())

                                          Resume_str extracted_skills
0           HR ADMINISTRATOR/MARKETING ASSOCIATE\...               []
1           HR SPECIALIST, US HR OPERATIONS      ...               []
2           HR DIRECTOR       Summary      Over 2...               []
3           HR SPECIALIST       Summary    Dedica...               []
4           HR MANAGER         Skill Highlights  ...               []


In [9]:
from sentence_transformers import SentenceTransformer, util

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')
print('model ready')

# Fixed typo: spelled 'job_description' correctly in the arguments
def rank_resumes(job_description, resume_df):
    # Encode the job description
    jd_embedding = model.encode(job_description, convert_to_tensor=True)

    # Fixed column target: changed 'Resume' to 'Resume_str'
    resume_embeddings = model.encode(resume_df['Resume_str'].tolist(), convert_to_tensor=True)

    # Compute similarity scores
    cos_scores = util.cos_sim(jd_embedding, resume_embeddings)[0]

    # Assign scores back to the dataframe
    resume_df['match_score'] = cos_scores.cpu().numpy()

    # Cleaned up: removed the duplicate empty 'return' statement below this line
    return resume_df.sort_values(by='match_score', ascending=False)

# Your sample job description
sample_jd = "Looking for a Python developer with experience in Machine Learning, Pandas, Scikit-learn, and building web apps using Streamlit."

# Get ranked results
ranked_results = rank_resumes(sample_jd, df)
print(ranked_results[['Category', 'match_score']].head(10))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model ready
                    Category  match_score
926              AGRICULTURE     0.530303
291   INFORMATION-TECHNOLOGY     0.465646
128                 DESIGNER     0.454342
297   INFORMATION-TECHNOLOGY     0.423863
1111              CONSULTANT     0.421219
1932            CONSTRUCTION     0.417911
309   INFORMATION-TECHNOLOGY     0.417628
311   INFORMATION-TECHNOLOGY     0.410482
2395                AVIATION     0.408735
2242                 BANKING     0.401884


In [32]:
import torch

# 1. Automatically use GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"Model ready on device: {device}")

def rank_resumes(job_description, resume_df):
    jd_embedding = model.encode(job_description, convert_to_tensor=True)

    # 2. Added batch_size and show_progress_bar to optimize CPU/GPU rendering
    resume_embeddings = model.encode(
        resume_df['Resume_str'].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_tensor=True
    )

    cos_scores = util.cos_sim(jd_embedding, resume_embeddings)[0]
    resume_df['match_score'] = cos_scores.cpu().numpy()

    return resume_df.sort_values(by='match_score', ascending=False)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model ready on device: cpu


In [33]:
#Gap Analysis (Highlighting Missing Skills)

def gap_analysis(job_skills, candidate_skills):
    job_set = set(job_skills)
    candidate_set = set(candidate_skills)

    missing_skills = job_set - candidate_set
    matched_skills = job_set.intersection(candidate_set)

    return {
        "matched": list(matched_skills),
        "missing": list(missing_skills)
    }

top_candidate_skills = ranked_results.iloc[0]['extracted_skills']
jd_skills = ['python', 'machine learning', 'pandas', 'scikit-learn', 'docker'] # required by JD

analysis = gap_analysis(jd_skills, top_candidate_skills)
print("Matched Skills:", analysis['matched'])
print("Missing Skills:", analysis['missing'])

Matched Skills: ['pandas', 'scikit-learn', 'python']
Missing Skills: ['docker', 'machine learning']


In [ ]:
import pandas as pd
import re
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# 1. Initialize Pipeline & Data Setup
device = "cpu"
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Using sample data structure to match exact dataset format
data = {
    'ID': ['16852973', '22323967', '33176873'],
    'Category': ['HR', 'HR', 'HR'],
    'Resume_str': [
        "HR ADMINISTRATOR with talent acquisition, onboarding experience.",
        "HR SPECIALIST, US HR OPERATIONS handling employee relations.",
        "HR DIRECTOR. Core skills include performance management and strategy."
    ]
}
df = pd.DataFrame(data)

# Master skill keywords database
MASTER_SKILL_DB = [
    'python', 'java', 'c++', 'machine learning', 'deep learning', 'sql', 'pandas',
    'numpy', 'scikit-learn', 'react', 'node.js', 'mongodb', 'html', 'css',
    'javascript', 'flask', 'streamlit', 'docker', 'nlp', 'pytorch', 'tensorflow',
    'recruitment', 'onboarding', 'talent acquisition', 'payroll', 'hris',
    'employee relations', 'performance management', 'training', 'conflict resolution',
    'accounting', 'budgeting', 'financial analysis', 'tax', 'auditing', 'excel',
    'sales', 'marketing', 'negotiation', 'strategy', 'leadership', 'project management'
]

def clean_text(text):
    """ ✔ Added: Resume text cleaning & preprocessing """
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]*>', ' ', text)  # Remove HTML artifacts if present
    text = re.sub(r'[\r\n\t]+', ' ', text) # Flatten layout spacing breaks
    text = re.sub(r'[^\w\s\-\.\,\/]', '', text) # Strip strange symbolic code breaks
    text = re.sub(r'\s+', ' ', text) # Standardize whitespace spans
    return text.strip().lower()

def extract_skills(text, skill_database):
    """ ✔ Skill extraction using NLP """
    text_cleaned = clean_text(text)
    found_skills = [skill for skill in skill_database if re.search(r'\b' + re.escape(skill) + r'\b', text_cleaned)]
    return list(set(found_skills))

# 2. Build Interactive Notebook Widgets
jd_input = widgets.Textarea(
    value="Looking for an HR professional experienced in talent acquisition, onboarding, employee relations, and performance management.",
    placeholder='Paste Job Description Here...',
    description='Job Desc:',
    layout=widgets.Layout(width='90%', height='100px')
)

skills_input = widgets.Text(
    value="recruitment, onboarding, talent acquisition, hr",
    placeholder='Comma-separated skills',
    description='Req Skills:',
    layout=widgets.Layout(width='90%')
)

# 🌟 Added: Optional Bonus: Weighting important skills widget
weights_input = widgets.Text(
    value="1.0, 2.0, 2.0, 0.5",
    placeholder='Comma-separated numbers matching your required skills above',
    description='Skill Weights:',
    layout=widgets.Layout(width='90%')
)

top_k_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    step=1,
    description='Top Candidates:',
    layout=widgets.Layout(width='50%')
)

btn_screen = widgets.Button(
    description='🚀 Screen & Rank Resumes',
    button_style='success',
    layout=widgets.Layout(width='30%', margin='10px 0px 10px 0px')
)

output_area = widgets.Output()

# 3. Core Processing Logic
def handle_screening(b):
    with output_area:
        clear_output()
        print("🔄 Processing similarities and running keyword check...")

        # ✔ Job description parsing
        cleaned_jd = clean_text(jd_input.value)
        jd_skills = [s.strip().lower() for s in skills_input.value.split(',') if s.strip()]
        top_k = top_k_slider.value

        # 🌟 Parse custom weights out cleanly
        try:
            raw_weights = [float(w.strip()) for w in weights_input.value.split(',')]
            if len(raw_weights) < len(jd_skills):
                raw_weights += [1.0] * (len(jd_skills) - len(raw_weights))
            skill_weights = dict(zip(jd_skills, raw_weights[:len(jd_skills)]))
        except ValueError:
            skill_weights = {s: 1.0 for s in jd_skills}
            print("⚠️ Weight input parsing error. Defaulting to standard weight distributions.")

        # Calculate Embeddings with Preprocessed Text
        jd_embedding = model.encode(cleaned_jd, convert_to_tensor=True)

        # ✔ Text cleaning pipeline application to the main dataframe
        df['cleaned_resume'] = df['Resume_str'].apply(clean_text)
        resume_texts = df['cleaned_resume'].tolist()
        resume_embeddings = model.encode(resume_texts, batch_size=32, convert_to_tensor=True)

        # ✔ Resume-to-role similarity scoring
        cos_scores = util.cos_sim(jd_embedding, resume_embeddings)[0]

        # Extract keywords using normalized engine
        df['extracted_skills'] = df['Resume_str'].apply(lambda x: extract_skills(x, MASTER_SKILL_DB))

        # Compute dynamic scores combining vector match + customized skill weight multipliers
        final_scores = []
        breakdowns = []

        for idx, row in df.iterrows():
            candidate_skills = row['extracted_skills']

            # ✔ Skill gap identification matrix
            matched = list(set(jd_skills).intersection(set(candidate_skills)))
            missing = list(set(jd_skills) - set(candidate_skills))

            # Weight scoring ratio formulation
            total_possible_weight = sum(skill_weights.values()) if skill_weights else 1
            earned_weight = sum([skill_weights.get(sk, 1.0) for sk in matched])
            keyword_ratio = (earned_weight / total_possible_weight) if total_possible_weight > 0 else 0

            # Hybrid Calculation Balance: 70% Semantic Space context + 30% Keyword Matching profile
            semantic_score = cos_scores[idx].item() * 100
            keyword_score = keyword_ratio * 100
            hybrid_score = (semantic_score * 0.7) + (keyword_score * 0.3)

            final_scores.append(hybrid_score)
            breakdowns.append({'matched': matched, 'missing': missing})

        df['match_score'] = final_scores
        df['breakdown'] = breakdowns

        # ✔ Candidate ranking based on role fit
        ranked_df = df.sort_values(by='match_score', ascending=False).reset_index(drop=True)
        limit_k = min(top_k, len(ranked_df))

        # 🌟 Added: Optional Bonus: Visual comparison table construction
        html_output = f"<h3>🏆 Evaluation Matrix (Top {limit_k} Profiles Ranked)</h3>"
        html_output += '<table style="width:100%; border-collapse: collapse; margin-bottom: 20px; font-family: Arial, sans-serif; font-size:13px;">'
        html_output += '<tr style="background-color: #2b3e50; color: white; text-align: left;">'
        html_output += '<th style="padding: 8px; border: 1px solid #ddd;">Rank</th>'
        html_output += '<th style="padding: 8px; border: 1px solid #ddd;">Candidate ID</th>'
        html_output += '<th style="padding: 8px; border: 1px solid #ddd;">Category</th>'
        html_output += '<th style="padding: 8px; border: 1px solid #ddd;">Hybrid Score</th>'
        html_output += '<th style="padding: 8px; border: 1px solid #ddd;">Keywords Found</th>'
        html_output += '</tr>'

        for i in range(limit_k):
            row = ranked_df.iloc[i]
            bg = "#f9f9f9" if i % 2 == 0 else "#ffffff"
            matched_text = ', '.join(row['breakdown']['matched']) if row['breakdown']['matched'] else 'None'
            html_output += f'<tr style="background-color: {bg};">'
            html_output += f'<td style="padding: 8px; border: 1px solid #ddd; font-weight:bold;">{i+1}</td>'
            html_output += f'<td style="padding: 8px; border: 1px solid #ddd;">{row["ID"]}</td>'
            html_output += f'<td style="padding: 8px; border: 1px solid #ddd;">{row["Category"]}</td>'
            html_output += f'<td style="padding: 8px; border: 1px solid #ddd; color:green; font-weight:bold;">{row["match_score"]:.2f}%</td>'
            html_output += f'<td style="padding: 8px; border: 1px solid #ddd;">{matched_text}</td>'
            html_output += '</tr>'
        html_output += '</table>'

        # Deep Detailed Profile Layout Display Block
        for i in range(limit_k):
            row = ranked_df.iloc[i]
            candidate_skills = row['extracted_skills']
            bd = row['breakdown']

            # Formulate text displays showing dynamic configurations
            matched_weighted_str = ', '.join([f"{m} (x{skill_weights.get(m, 1.0)})" for m in bd['matched']]) if bd['matched'] else 'None'
            missing_str = ', '.join(bd['missing']) if bd['missing'] else 'None'

            html_output += f"""
            <div style="border: 1px solid #ddd; padding: 15px; margin-bottom: 10px; border-radius: 5px; background-color: #fcfcfc;">
                <b style="color: #2e7d32;">Rank {i+1} | Category: {row['Category']} | Overall Match: {row['match_score']:.2f}%</b><br>
                <b>Candidate ID:</b> {row['ID']}<br>
                <b>Extracted Skills Matrix:</b> <code>{', '.join(candidate_skills) if candidate_skills else 'None detected'}</code><br>
                <span style="color: green;">✅ <b>Matched Required Skills (With Weights):</b> {matched_weighted_str}</span><br>
                <span style="color: red;">❌ <b>Missing Required Skills Gap:</b> {missing_str}</span><br>
                <details style="margin-top: 5px;">
                    <summary style="cursor:pointer; color: #1565c0;">View Preprocessed Source Segment</summary>
                    <pre style="background: #f0f0f0; padding: 6px; font-size: 11px; white-space: pre-wrap;">{row['Resume_str'][:300]}...</pre>
                </details>
            </div>
            """
        display(HTML(html_output))


# Link the button click to execution
btn_screen.on_click(handle_screening)

# 4. Render Layout directly in the cell
display(widgets.VBox([
    widgets.Label(value="📄 In-Notebook Resume Screening Dashboard", style={'font_weight': 'bold'}),
    jd_input,
    skills_input,
    top_k_slider,
    btn_screen,
    output_area
]))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]